In [2]:
# ============================================================
# STUDYMATE AI
# Intelligent Personalized Learning & Study Assistant
# CELL 1 — INSTALLATION
# ============================================================

!pip -q install "gradio==5.23.3" gTTS

print("=" * 70)
print("📚 STUDYMATE AI")
print("=" * 70)
print("✅ Gradio installed")
print("✅ Text-to-Speech installed")
print("🚀 Environment ready")
print("=" * 70)

📚 STUDYMATE AI
✅ Gradio installed
✅ Text-to-Speech installed
🚀 Environment ready


In [3]:
# ============================================================
# STUDYMATE AI
# CELL 2 — STUDY ENGINE
# ============================================================

import gradio as gr
import re
import random
import os
import tempfile
from gtts import gTTS

# ------------------------------------------------------------
# SUBJECT DATABASE
# ------------------------------------------------------------

SUBJECTS = {
    "DBMS": {
        "keywords": [
            "database", "dbms", "sql", "table", "normalization",
            "transaction", "primary key", "foreign key"
        ],
        "topics": [
            "Database Fundamentals",
            "ER Model",
            "Relational Model",
            "SQL",
            "Normalization",
            "Transactions"
        ]
    },

    "Computer Networks": {
        "keywords": [
            "network", "tcp", "ip", "osi", "http", "router",
            "switch", "protocol", "networking"
        ],
        "topics": [
            "Networking Basics",
            "OSI Model",
            "TCP/IP",
            "Routing",
            "Network Protocols",
            "Network Security"
        ]
    },

    "Operating Systems": {
        "keywords": [
            "operating system", "os", "process", "thread",
            "deadlock", "memory", "scheduling", "linux"
        ],
        "topics": [
            "OS Fundamentals",
            "Process Management",
            "CPU Scheduling",
            "Deadlocks",
            "Memory Management",
            "File Systems"
        ]
    },

    "Python": {
        "keywords": [
            "python", "function", "list", "tuple", "dictionary",
            "class", "object", "loop", "exception"
        ],
        "topics": [
            "Python Basics",
            "Variables and Data Types",
            "Control Statements",
            "Functions",
            "Collections",
            "OOP"
        ]
    },

    "Data Structures": {
        "keywords": [
            "array", "stack", "queue", "linked list", "tree",
            "graph", "sorting", "searching", "data structure"
        ],
        "topics": [
            "Arrays",
            "Linked Lists",
            "Stacks",
            "Queues",
            "Trees",
            "Graphs"
        ]
    },

    "Software Engineering": {
        "keywords": [
            "software engineering", "sdlc", "agile", "waterfall",
            "testing", "requirement", "software"
        ],
        "topics": [
            "Software Engineering Basics",
            "SDLC",
            "Requirement Analysis",
            "Software Design",
            "Testing",
            "Agile"
        ]
    }
}


# ------------------------------------------------------------
# SAMPLE QUESTION BANK
# ------------------------------------------------------------

QUESTION_BANK = {

    "DBMS": [
        "What is DBMS and what are its advantages?",
        "Explain the difference between primary key and foreign key.",
        "What is normalization? Explain its different forms.",
        "What is a transaction in DBMS?",
        "Explain ACID properties.",
        "What is SQL? Explain DDL and DML."
    ],

    "Computer Networks": [
        "Explain the OSI model and its seven layers.",
        "What is the difference between TCP and UDP?",
        "What is an IP address?",
        "Explain the functions of a router.",
        "What are network protocols?",
        "Explain HTTP and HTTPS."
    ],

    "Operating Systems": [
        "What is an operating system?",
        "Explain process and thread.",
        "What is CPU scheduling?",
        "Explain deadlock and its necessary conditions.",
        "What is virtual memory?",
        "Explain different types of operating systems."
    ],

    "Python": [
        "What are the main features of Python?",
        "Explain lists, tuples and dictionaries.",
        "What is a function in Python?",
        "Explain object-oriented programming in Python.",
        "What is exception handling?",
        "Explain Python loops."
    ],

    "Data Structures": [
        "What is a data structure?",
        "Explain stack and its applications.",
        "Explain queue and its applications.",
        "What is a linked list?",
        "Explain binary trees.",
        "What is the difference between linear and non-linear data structures?"
    ],

    "Software Engineering": [
        "What is software engineering?",
        "Explain the SDLC.",
        "What is requirement analysis?",
        "Compare Agile and Waterfall models.",
        "What is software testing?",
        "Explain functional and non-functional requirements."
    ]
}


# ------------------------------------------------------------
# TOPIC DETECTION
# ------------------------------------------------------------

def detect_subject(text):

    text_lower = text.lower()

    scores = {}

    for subject, data in SUBJECTS.items():

        score = 0

        for keyword in data["keywords"]:

            if keyword in text_lower:
                score += 1

        scores[subject] = score

    best_subject = max(scores, key=scores.get)

    if scores[best_subject] == 0:
        return "General Study"

    return best_subject


# ------------------------------------------------------------
# TEXT CLEANING
# ------------------------------------------------------------

def clean_text(text):

    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


# ------------------------------------------------------------
# SUMMARY GENERATOR
# ------------------------------------------------------------

def generate_summary(text):

    text = clean_text(text)

    if not text:

        return "Please enter your study material first."

    sentences = re.split(r'(?<=[.!?])\s+', text)

    important = sentences[:5]

    summary = "\n\n".join(
        f"• {sentence}"
        for sentence in important
        if sentence.strip()
    )

    return summary


# ------------------------------------------------------------
# IMPORTANT QUESTIONS
# ------------------------------------------------------------

def generate_questions(text):

    subject = detect_subject(text)

    if subject in QUESTION_BANK:

        questions = QUESTION_BANK[subject]

    else:

        questions = [
            "What is the main concept discussed in this topic?",
            "Explain the important terms related to this topic.",
            "What are the advantages and applications?",
            "Explain the working process with an example.",
            "What are the important points to remember?",
            "Compare the major concepts discussed."
        ]

    html = ""

    for i, question in enumerate(questions, 1):

        html += f"""
        <div class="question-card">
            <span class="q-number">Q{i}</span>
            <span>{question}</span>
        </div>
        """

    return html


# ------------------------------------------------------------
# FLASHCARDS
# ------------------------------------------------------------

def generate_flashcards(text):

    subject = detect_subject(text)

    if subject == "DBMS":

        cards = [
            ("What is DBMS?", "Software used to create, manage and organize databases."),
            ("What is a primary key?", "A field that uniquely identifies each record."),
            ("What is normalization?", "A technique used to reduce data redundancy."),
            ("What is SQL?", "Structured Query Language used to work with relational databases.")
        ]

    elif subject == "Computer Networks":

        cards = [
            ("What is TCP?", "A reliable connection-oriented communication protocol."),
            ("What is IP?", "Internet Protocol responsible for addressing and routing."),
            ("What is OSI?", "A seven-layer reference model for network communication."),
            ("What is HTTP?", "A protocol used for communication between web clients and servers.")
        ]

    elif subject == "Operating Systems":

        cards = [
            ("What is a process?", "A program that is currently being executed."),
            ("What is a thread?", "The smallest unit of execution within a process."),
            ("What is deadlock?", "A situation where processes wait indefinitely for resources."),
            ("What is virtual memory?", "A memory management technique using disk space as an extension of RAM.")
        ]

    else:

        cards = [
            ("Key Concept", "Review the main definition and purpose."),
            ("Important Term", "Understand its meaning and practical application."),
            ("Advantages", "Remember the major benefits."),
            ("Application", "Know at least one real-world use case.")
        ]

    html = '<div class="flash-grid">'

    for question, answer in cards:

        html += f"""
        <div class="flashcard">
            <div class="flash-question">❓ {question}</div>
            <div class="flash-answer">{answer}</div>
        </div>
        """

    html += "</div>"

    return html


# ------------------------------------------------------------
# STUDY PLAN
# ------------------------------------------------------------

def generate_study_plan(text, days):

    subject = detect_subject(text)

    topics = SUBJECTS.get(
        subject,
        {
            "topics": [
                "Introduction",
                "Important Concepts",
                "Core Topics",
                "Examples",
                "Revision",
                "Practice"
            ]
        }
    )["topics"]

    days = int(days)

    html = '<div class="plan-grid">'

    for day in range(1, days + 1):

        topic = topics[(day - 1) % len(topics)]

        html += f"""
        <div class="plan-card">
            <div class="day">DAY {day}</div>
            <h3>{topic}</h3>
            <p>📖 Learn concept</p>
            <p>📝 Make short notes</p>
            <p>🧠 Practice questions</p>
            <p>🔄 Quick revision</p>
        </div>
        """

    html += "</div>"

    return html


# ------------------------------------------------------------
# WEAK TOPIC ANALYSIS
# ------------------------------------------------------------

def weak_topics(text):

    subject = detect_subject(text)

    if subject in SUBJECTS:

        topics = SUBJECTS[subject]["topics"]

    else:

        topics = [
            "Basic Concepts",
            "Important Definitions",
            "Practical Applications",
            "Problem Solving",
            "Revision"
        ]

    html = ""

    for topic in topics[:5]:

        html += f"""
        <div class="weak-item">
            <span>🎯</span>
            <div>
                <strong>{topic}</strong>
                <p>Recommended for additional revision and practice.</p>
            </div>
        </div>
        """

    return html


# ------------------------------------------------------------
# DASHBOARD
# ------------------------------------------------------------

def dashboard(text, days):

    subject = detect_subject(text)

    word_count = len(text.split())

    return f"""
    <div class="dashboard">

        <div class="stat-card">
            <div class="stat-icon">📚</div>
            <div>
                <span>SUBJECT</span>
                <strong>{subject}</strong>
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-icon">📝</div>
            <div>
                <span>WORDS</span>
                <strong>{word_count}</strong>
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-icon">📅</div>
            <div>
                <span>STUDY DAYS</span>
                <strong>{days}</strong>
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-icon">🎯</div>
            <div>
                <span>MODE</span>
                <strong>Personalized</strong>
            </div>
        </div>

    </div>
    """


# ------------------------------------------------------------
# COMPLETE ANALYSIS
# ------------------------------------------------------------

def analyze_material(text, days):

    if not text or not text.strip():

        message = """
        <div class="empty-box">
            📚 Please enter your study material to generate your study plan.
        </div>
        """

        return (
            message,
            message,
            message,
            message,
            message,
            message
        )

    summary = generate_summary(text)

    questions = generate_questions(text)

    flashcards = generate_flashcards(text)

    plan = generate_study_plan(text, days)

    weak = weak_topics(text)

    stats = dashboard(text, days)

    return summary, questions, flashcards, plan, weak, stats


# ------------------------------------------------------------
# TEXT TO SPEECH
# ------------------------------------------------------------

def read_notes(text):

    if not text or not text.strip():

        return None

    try:

        file_path = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".mp3"
        ).name

        speech = gTTS(
            text=text[:3000],
            lang="en"
        )

        speech.save(file_path)

        return file_path

    except Exception as e:

        print("TTS Error:", e)

        return None


print("=" * 70)
print("📚 STUDYMATE AI ENGINE")
print("=" * 70)
print("✅ Subject detection ready")
print("✅ Summary generator ready")
print("✅ Question generator ready")
print("✅ Flashcard generator ready")
print("✅ Study planner ready")
print("✅ Weak-topic analyzer ready")
print("✅ Text-to-Speech ready")
print("=" * 70)
print("🚀 Cell 2 completed successfully")

📚 STUDYMATE AI ENGINE
✅ Subject detection ready
✅ Summary generator ready
✅ Question generator ready
✅ Flashcard generator ready
✅ Study planner ready
✅ Weak-topic analyzer ready
✅ Text-to-Speech ready
🚀 Cell 2 completed successfully


In [5]:
# ============================================================
# STUDYMATE AI
# CELL 3 — FULL PROFESSIONAL COLOR UI
# ============================================================

import gradio as gr


# ============================================================
# SAMPLE MATERIAL
# ============================================================

SAMPLE_TEXT = """
Database Management System (DBMS) is a software system used
to create, store, organize, retrieve and manage data efficiently.

A DBMS provides data security, data consistency, data integrity,
backup and recovery. It is widely used in banking, hospitals,
education, e-commerce and business applications.

A relational database stores data in tables consisting of rows
and columns. Each row represents a record and each column
represents an attribute.

A primary key uniquely identifies each record in a table.
A foreign key connects one table with another table.

Normalization is the process of organizing data to reduce
redundancy and improve data integrity. The major normal forms
are First Normal Form, Second Normal Form and Third Normal Form.

SQL stands for Structured Query Language. SQL is used to create,
retrieve, update and delete data from relational databases.

SQL commands include DDL, DML, DQL, DCL and TCL.

Transactions are logical units of database operations.
Transactions follow ACID properties: Atomicity, Consistency,
Isolation and Durability.

Database security protects information from unauthorized access.
Authentication verifies users and authorization controls their
permissions.

Indexes improve data retrieval performance. Views provide
virtual representations of data. Stored procedures contain
reusable SQL statements.

DBMS technology is essential in modern software applications
because it provides an efficient and secure way to manage
large amounts of information.
"""


# ============================================================
# PROFESSIONAL CSS
# ============================================================

CSS = """

/* ==========================================================
   MAIN PAGE
   ========================================================== */

body {
    margin: 0 !important;

    background:
        radial-gradient(
            circle at 10% 10%,
            rgba(80, 110, 255, 0.25),
            transparent 30%
        ),

        radial-gradient(
            circle at 90% 20%,
            rgba(170, 70, 255, 0.25),
            transparent 30%
        ),

        linear-gradient(
            135deg,
            #09142f 0%,
            #10265b 35%,
            #24105a 70%,
            #071c38 100%
        ) !important;

    min-height: 100vh;
}


/* ==========================================================
   GRADIO MAIN CONTAINER
   ========================================================== */

.gradio-container {
    max-width: 1450px !important;

    margin: auto !important;

    padding:
        25px 30px 40px 30px !important;

    background: transparent !important;
}


/* ==========================================================
   HERO
   ========================================================== */

.hero {
    position: relative;

    overflow: hidden;

    padding: 48px 35px;

    margin-bottom: 25px;

    border-radius: 30px;

    background:
        linear-gradient(
            135deg,
            #2864f0 0%,
            #5146e5 42%,
            #8b3fd9 75%,
            #c13acb 100%
        );

    border:
        1px solid rgba(255,255,255,0.25);

    box-shadow:
        0 25px 70px rgba(0,0,0,0.35);

    text-align: center;

    color: white;
}


/* decorative glow */

.hero::before {

    content: "";

    position: absolute;

    width: 280px;
    height: 280px;

    border-radius: 50%;

    background:
        rgba(255,255,255,0.12);

    top: -150px;
    right: -80px;

}

.hero::after {

    content: "";

    position: absolute;

    width: 220px;
    height: 220px;

    border-radius: 50%;

    background:
        rgba(255,255,255,0.08);

    bottom: -130px;
    left: -70px;

}


.hero h1 {

    position: relative;

    z-index: 2;

    margin: 0;

    font-size: 48px;

    font-weight: 900;

    color: white !important;

    letter-spacing: -1px;

}


.hero p {

    position: relative;

    z-index: 2;

    color: rgba(255,255,255,0.95) !important;

    font-size: 17px;

    line-height: 1.7;

    margin:
        8px auto 0 auto;

    max-width: 850px;

}


/* ==========================================================
   TABS
   ========================================================== */

.tabs {

    background:
        rgba(10, 22, 55, 0.85) !important;

    border:
        1px solid rgba(255,255,255,0.12) !important;

    border-radius: 22px !important;

    padding: 8px !important;

    box-shadow:
        0 15px 45px rgba(0,0,0,0.25) !important;

}


/* tab buttons */

button[role="tab"] {

    color:
        #d9e5ff !important;

    background:
        transparent !important;

    border-radius:
        13px !important;

    font-weight:
        700 !important;

    font-size:
        14px !important;

}


button[role="tab"]:hover {

    background:
        rgba(255,255,255,0.08) !important;

}


button[role="tab"][aria-selected="true"] {

    color:
        white !important;

    background:
        linear-gradient(
            135deg,
            #3475ff,
            #7448e8
        ) !important;

    box-shadow:
        0 8px 25px rgba(70,90,240,0.35);

}


/* ==========================================================
   SECTION HEADER
   ========================================================== */

.section-title {

    margin:
        25px 0 15px 0;

    padding:
        5px 5px 5px 2px;

}


.section-title h2 {

    color:
        #ffffff !important;

    font-size:
        25px !important;

    font-weight:
        800 !important;

    margin-bottom:
        5px !important;

}


.section-title p {

    color:
        #b8c9ef !important;

    font-size:
        14px !important;

}


/* ==========================================================
   INPUT PANEL
   ========================================================== */

.input-panel {

    background:
        linear-gradient(
            145deg,
            rgba(31,55,110,0.95),
            rgba(52,31,103,0.95)
        );

    border:
        1px solid rgba(150,170,255,0.22);

    border-radius:
        22px;

    padding:
        20px;

    box-shadow:
        0 15px 40px rgba(0,0,0,0.25);

}


/* ==========================================================
   TEXTBOX
   ========================================================== */

textarea,
input {

    color:
        #eef4ff !important;

    background:
        #142650 !important;

    border:
        1px solid #3b5791 !important;

}


textarea {

    border-radius:
        16px !important;

    font-size:
        15px !important;

    line-height:
        1.75 !important;

}


textarea::placeholder,
input::placeholder {

    color:
        #8fa5d5 !important;

}


/* labels */

label {

    color:
        #dce8ff !important;

    font-weight:
        700 !important;

}


/* ==========================================================
   SLIDER
   ========================================================== */

input[type="range"] {

    accent-color:
        #7355e8 !important;

}


/* ==========================================================
   BUTTONS
   ========================================================== */

.primary-btn {

    background:
        linear-gradient(
            135deg,
            #3478ff,
            #7447e8,
            #a53fd1
        ) !important;

    color:
        white !important;

    border:
        none !important;

    border-radius:
        13px !important;

    font-weight:
        800 !important;

    box-shadow:
        0 8px 25px rgba(85,75,230,0.35) !important;

}


.primary-btn:hover {

    transform:
        translateY(-1px);

}


.secondary-btn {

    background:
        linear-gradient(
            135deg,
            #172f63,
            #342263
        ) !important;

    color:
        #dce8ff !important;

    border:
        1px solid #4a5fa1 !important;

    border-radius:
        13px !important;

    font-weight:
        700 !important;

}


/* ==========================================================
   DASHBOARD
   ========================================================== */

.dashboard {

    display:
        grid;

    grid-template-columns:
        repeat(4, 1fr);

    gap:
        15px;

    margin:
        10px 0 25px 0;

}


.stat-card {

    position:
        relative;

    overflow:
        hidden;

    display:
        flex;

    align-items:
        center;

    gap:
        15px;

    padding:
        21px;

    min-height:
        90px;

    border-radius:
        19px;

    background:
        linear-gradient(
            135deg,
            #183568,
            #342267
        );

    border:
        1px solid rgba(140,170,255,0.25);

    box-shadow:
        0 12px 30px rgba(0,0,0,0.25);

}


.stat-card::after {

    content: "";

    position:
        absolute;

    width:
        80px;

    height:
        80px;

    border-radius:
        50%;

    background:
        rgba(110,150,255,0.12);

    right:
        -25px;

    top:
        -25px;

}


.stat-icon {

    font-size:
        30px;

}


.stat-card span {

    display:
        block;

    color:
        #91a9d9 !important;

    font-size:
        10px;

    font-weight:
        800;

    letter-spacing:
        1px;

}


.stat-card strong {

    display:
        block;

    color:
        white !important;

    font-size:
        17px;

    margin-top:
        5px;

}


/* ==========================================================
   OUTPUT BOX
   ========================================================== */

.output-card {

    background:
        linear-gradient(
            145deg,
            rgba(24,48,95,0.96),
            rgba(44,25,82,0.96)
        );

    border:
        1px solid rgba(140,165,255,0.20);

    border-radius:
        21px;

    padding:
        22px;

    margin-bottom:
        16px;

    box-shadow:
        0 14px 35px rgba(0,0,0,0.24);

}


.output-card h3 {

    color:
        #ffffff !important;

    margin-top:
        0 !important;

}


/* ==========================================================
   SUMMARY
   ========================================================== */

.summary-box {

    background:
        linear-gradient(
            135deg,
            #1a4385,
            #4b2b88
        );

    border:
        1px solid rgba(170,190,255,0.25);

    border-left:
        5px solid #62a5ff;

    border-radius:
        16px;

    padding:
        22px;

    color:
        #edf4ff !important;

    line-height:
        1.8;

}


/* ==========================================================
   QUESTIONS
   ========================================================== */

.question-card {

    display:
        flex;

    align-items:
        center;

    gap:
        14px;

    background:
        linear-gradient(
            135deg,
            #142f61,
            #302064
        );

    border:
        1px solid rgba(130,160,255,0.20);

    padding:
        15px;

    margin:
        9px 0;

    border-radius:
        14px;

    color:
        #edf4ff !important;

    font-weight:
        600;

}


.q-number {

    min-width:
        39px;

    height:
        39px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        10px;

    background:
        linear-gradient(
            135deg,
            #3e83ff,
            #9149df
        );

    color:
        white;

    font-weight:
        900;

}


/* ==========================================================
   FLASHCARDS
   ========================================================== */

.flash-grid {

    display:
        grid;

    grid-template-columns:
        repeat(2, 1fr);

    gap:
        16px;

}


.flashcard {

    background:
        linear-gradient(
            145deg,
            #17366d,
            #39246e
        );

    border:
        1px solid rgba(145,170,255,0.22);

    border-radius:
        18px;

    padding:
        21px;

    min-height:
        120px;

    box-shadow:
        0 10px 30px rgba(0,0,0,0.20);

}


.flash-question {

    color:
        #8dc1ff !important;

    font-weight:
        800;

    margin-bottom:
        12px;

}


.flash-answer {

    color:
        #d9e5ff !important;

    line-height:
        1.6;

}


/* ==========================================================
   STUDY PLAN
   ========================================================== */

.plan-grid {

    display:
        grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap:
        15px;

}


.plan-card {

    background:
        linear-gradient(
            145deg,
            #18396f,
            #40266f
        );

    border:
        1px solid rgba(145,170,255,0.20);

    border-radius:
        18px;

    padding:
        20px;

    box-shadow:
        0 10px 28px rgba(0,0,0,0.20);

}


.plan-card .day {

    color:
        #73b0ff !important;

    font-size:
        12px;

    font-weight:
        900;

    letter-spacing:
        1px;

}


.plan-card h3 {

    color:
        white !important;

}


.plan-card p {

    color:
        #c4d2ef !important;

    margin:
        6px 0;

}


/* ==========================================================
   WEAK TOPICS
   ========================================================== */

.weak-item {

    display:
        flex;

    gap:
        14px;

    background:
        linear-gradient(
            135deg,
            #493b1d,
            #3e2d19
        );

    border:
        1px solid rgba(255,210,100,0.22);

    border-radius:
        15px;

    padding:
        16px;

    margin:
        10px 0;

}


.weak-item span {

    font-size:
        22px;

}


.weak-item strong {

    color:
        #ffd878 !important;

}


.weak-item p {

    color:
        #ddcda4 !important;

    margin:
        5px 0 0 0;

}


/* ==========================================================
   EMPTY MESSAGE
   ========================================================== */

.empty-box {

    background:
        linear-gradient(
            135deg,
            #183568,
            #39236d
        );

    border:
        1px dashed #6f8dcc;

    border-radius:
        17px;

    padding:
        25px;

    text-align:
        center;

    color:
        #dce8ff !important;

}


/* ==========================================================
   MARKDOWN
   ========================================================== */

.prose,
.markdown-body {

    color:
        #dce8ff !important;

}


.prose p,
.prose li {

    color:
        #c8d5ee !important;

    line-height:
        1.7;

}


.prose strong {

    color:
        white !important;

}


/* ==========================================================
   AUDIO
   ========================================================== */

audio {

    width:
        100%;

}


/* ==========================================================
   FOOTER
   ========================================================== */

.footer {

    text-align:
        center;

    padding:
        30px 10px 15px 10px;

    color:
        #8fa5d5 !important;

    font-size:
        13px;

}


.footer b {

    color:
        #b8d5ff !important;

}


/* ==========================================================
   MOBILE
   ========================================================== */

@media(max-width: 1000px) {

    .dashboard {

        grid-template-columns:
            repeat(2, 1fr);

    }

    .plan-grid {

        grid-template-columns:
            repeat(2, 1fr);

    }

}


@media(max-width: 700px) {

    .gradio-container {

        padding:
            15px !important;

    }

    .hero {

        padding:
            32px 18px;

        border-radius:
            22px;

    }

    .hero h1 {

        font-size:
            34px;

    }

    .dashboard,
    .plan-grid,
    .flash-grid {

        grid-template-columns:
            1fr;

    }

}

"""


# ============================================================
# CREATE APPLICATION
# ============================================================

with gr.Blocks(
    css=CSS,
    title="StudyMate AI"
) as app:

    # --------------------------------------------------------
    # HERO
    # --------------------------------------------------------

    gr.HTML("""
    <div class="hero">

        <h1>📚 StudyMate AI</h1>

        <p>
            Intelligent Personalized Learning & Study Assistant
        </p>

        <p>
            Transform your study material into smart summaries,
            important questions, flashcards, revision topics
            and personalized study plans.
        </p>

    </div>
    """)


    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():


        # ====================================================
        # TAB 1 — STUDY ANALYZER
        # ====================================================

        with gr.Tab("📖 Study Analyzer"):

            gr.HTML("""
            <div class="section-title">

                <h2>🧠 Analyze Your Study Material</h2>

                <p>
                    Paste your notes, textbook content, syllabus
                    or study material and generate personalized
                    learning resources.
                </p>

            </div>
            """)


            # INPUT PANEL

            gr.HTML("""
            <div class="input-panel">
            """)


            with gr.Row():

                with gr.Column(scale=4):

                    material = gr.Textbox(
                        label="📄 Study Material",
                        placeholder=
                        "Paste your study material here...",
                        lines=13
                    )


                with gr.Column(scale=1):

                    days = gr.Slider(
                        minimum=1,
                        maximum=10,
                        value=5,
                        step=1,
                        label="📅 Study Days"
                    )

                    analyze_btn = gr.Button(
                        "✨ Generate Study Plan",
                        elem_classes="primary-btn"
                    )

                    sample_btn = gr.Button(
                        "📝 Load Sample Material",
                        elem_classes="secondary-btn"
                    )


            gr.HTML("""
            </div>
            """)


            # ------------------------------------------------
            # DASHBOARD
            # ------------------------------------------------

            gr.HTML("""
            <div class="section-title">
                <h2>📊 Learning Dashboard</h2>
            </div>
            """)

            dashboard_output = gr.HTML()


            # ------------------------------------------------
            # SUMMARY + QUESTIONS
            # ------------------------------------------------

            with gr.Row():

                with gr.Column():

                    gr.HTML("""
                    <div class="output-card">
                        <h3>✨ Smart Summary</h3>
                    </div>
                    """)

                    summary_output = gr.Markdown()


                with gr.Column():

                    gr.HTML("""
                    <div class="output-card">
                        <h3>❓ Important Questions</h3>
                    </div>
                    """)

                    questions_output = gr.HTML()


            # ------------------------------------------------
            # FLASHCARDS + WEAK TOPICS
            # ------------------------------------------------

            with gr.Row():

                with gr.Column():

                    gr.HTML("""
                    <div class="output-card">
                        <h3>🃏 Smart Flashcards</h3>
                    </div>
                    """)

                    flashcards_output = gr.HTML()


                with gr.Column():

                    gr.HTML("""
                    <div class="output-card">
                        <h3>🎯 Topics to Revise</h3>
                    </div>
                    """)

                    weak_output = gr.HTML()


            # ------------------------------------------------
            # STUDY PLAN
            # ------------------------------------------------

            gr.HTML("""
            <div class="section-title">
                <h2>📅 Personalized Study Plan</h2>
            </div>
            """)

            plan_output = gr.HTML()


        # ====================================================
        # TAB 2 — LISTEN & REVISE
        # ====================================================

        with gr.Tab("🔊 Listen & Revise"):

            gr.HTML("""
            <div class="section-title">

                <h2>🔊 Audio Revision</h2>

                <p>
                    Convert your study notes into audio and
                    revise your material by listening.
                </p>

            </div>
            """)


            with gr.Row():

                with gr.Column():

                    audio_text = gr.Textbox(
                        label="📄 Notes",
                        placeholder=
                        "Enter the notes you want to listen to...",
                        lines=12
                    )

                    speak_btn = gr.Button(
                        "🔊 Generate Study Audio",
                        elem_classes="primary-btn"
                    )


                with gr.Column():

                    audio_output = gr.Audio(
                        label="🎧 Your Study Audio",
                        type="filepath"
                    )


            speak_btn.click(
                read_notes,
                inputs=audio_text,
                outputs=audio_output
            )


        # ====================================================
        # TAB 3 — HOW IT WORKS
        # ====================================================

        with gr.Tab("💡 How It Works"):

            gr.HTML("""
            <div class="section-title">

                <h2>🚀 How StudyMate AI Works</h2>

                <p>
                    A simple learning workflow designed to make
                    studying more organized and effective.
                </p>

            </div>
            """)


            gr.HTML("""

            <div class="plan-grid">

                <div class="plan-card">

                    <div class="day">STEP 01</div>

                    <h3>📄 INPUT</h3>

                    <p>
                        Enter your notes, syllabus or study material.
                    </p>

                </div>


                <div class="plan-card">

                    <div class="day">STEP 02</div>

                    <h3>🧠 ANALYZE</h3>

                    <p>
                        The system identifies the subject and
                        important learning concepts.
                    </p>

                </div>


                <div class="plan-card">

                    <div class="day">STEP 03</div>

                    <h3>✨ SUMMARIZE</h3>

                    <p>
                        Important information is converted into
                        concise revision notes.
                    </p>

                </div>


                <div class="plan-card">

                    <div class="day">STEP 04</div>

                    <h3>❓ PRACTICE</h3>

                    <p>
                        Important questions and flashcards help
                        students practice.
                    </p>

                </div>


                <div class="plan-card">

                    <div class="day">STEP 05</div>

                    <h3>📅 PLAN</h3>

                    <p>
                        A personalized study schedule is created
                        according to the selected days.
                    </p>

                </div>


                <div class="plan-card">

                    <div class="day">STEP 06</div>

                    <h3>🎯 REVISE</h3>

                    <p>
                        Important topics are highlighted for
                        focused revision.
                    </p>

                </div>

            </div>

            """)


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML("""
    <div class="footer">

        <b>📚 StudyMate AI</b>

        <br>

        Intelligent Personalized Learning & Study Assistant

        <br><br>

        Python • Gradio • NLP • Text Processing •
        Personalized Learning

    </div>
    """)


    # ========================================================
    # EVENTS
    # ========================================================

    analyze_btn.click(

        analyze_material,

        inputs=[
            material,
            days
        ],

        outputs=[
            summary_output,
            questions_output,
            flashcards_output,
            plan_output,
            weak_output,
            dashboard_output
        ]

    )


    sample_btn.click(

        lambda: SAMPLE_TEXT,

        outputs=material

    )


# ============================================================
# LAUNCH
# ============================================================

print("=" * 70)
print("📚 STUDYMATE AI")
print("=" * 70)
print("🚀 Starting professional interface...")
print("=" * 70)

app.launch(
    share=True,
    debug=True
)

📚 STUDYMATE AI
🚀 Starting professional interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ed2c9da7caf8354df.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2ed2c9da7caf8354df.gradio.live
